In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName('unit_test') \
    .config("spark.jars", "/opt/spark/jars/iceberg-spark-runtime-3.5_2.12-1.6.0.jar") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.spark_catalog.type", "hive") \
    .config("spark.sql.catalog.local.warehouse", "s3a://datalake/iceberg") \
    .getOrCreate()

#Ajuste de log WARN log para ERROR
spark.sparkContext.setLogLevel("ERROR")

In [ ]:
import unittest

class DataQualityTestSuite(unittest.TestCase):
    def __init__(self, methodName='runTest', target_table=None):
        super().__init__(methodName)
        self.target_table = target_table

    def test_column_unique(self):
        """ Test column unique values"""
        
        src= spark.sql(f"""
            SELECT COUNT(*) 
            FROM (SELECT product_id, COUNT(*) c FROM {self.target_table}
            GROUP BY product_id HAVING c > 1)
        """).collect()[0][0]
        
        expected = 0
        
        self.assertEqual(src, expected, f"Existem IDs duplicados")
        

    def test_column_not_null(self):
        """ Test if column not null"""
        
        src = spark.sql(f"""
            SELECT COUNT(*) as n 
            FROM {self.target_table} 
            WHERE product_name IS NULL
        """).collect()[0]["n"]
        
        expected = 0        
        
        self.assertEqual(src, expected, f" Existem product_name nulos")
        
    
    def test_schema_validations(self):
        """ Test table's schema"""
        
        src = spark.sql(f"""
            SELECT * FROM {self.target_table} 
        """).dtypes
    
        expected = [
            ('product_id', 'string'),
            ('product_name', 'string'),
            ('category', 'string'),
            ('price', 'string')
            ]

        self.assertListEqual(src, expected, f" Existem product_name nulos")
        

In [ ]:
# Fábrica para criar uma nova classe de teste com o target_table embutido (injeção de dependência com class factory)

def create_test_suite_for_table(target_table):
    
    class PatchedTestSuite(DataQualityTestSuite):
        def __init__(self, methodName='runTest'):
            super().__init__(methodName=methodName, target_table=target_table)
    PatchedTestSuite.__name__ = f"DataQualityTestSuite_{target_table.replace('.', '_')}"
    return unittest.TestLoader().loadTestsFromTestCase(PatchedTestSuite)

In [ ]:
target_table='iceberg.bronze.tbl_bronze_product_catalog'

metrics_quality_suite = create_test_suite_for_table(target_table)
tests = unittest.TestSuite([metrics_quality_suite])
runner = unittest.TextTestRunner(verbosity=2)
runner.run(tests)